# South Barnegat Bay Onshore Wind Model Prediction with the Use of Long Short-Term Memory Neural Networks - Terminal Program for Real-Time Predictions

### Written by Krupam Patel
### Written in Python 3.11.14

##### Damp South and Western onshore winds initiate the summertime event known as upwelling. Upwelling in small, localized areas like Barnegat Bay has a significant impact on bay temperature, creating a large land-sea temperature difference. This difference can lead to harsh and fast onshore breezes that can “swamp” small watercraft. This study uses an LSTM, a deep learning neural network, to predict these large gusts, along with Naive and Linear Regression algorithms to verify its effectiveness. To utilize these models, thirteen variables were collected on an hourly basis for June-August. Once the models were created and trained, the mean absolute error was calculated for each of the models as a comparison. Shockingly, it seemed that Linear Regressions and Naive models performed marginally better than the LSTM, which had collapsed to predicting a value close to the mean in almost all tests. Accurate wind speeds and direction were still predicted, as the hypothesis says, just with different models. With the rarity of upwelling events and their spontaneity, it’s a wonder that the models could predict them in any way.

In [72]:
from joblib import dump, load
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression
from datetime import datetime
import time
import sys
import pandas as pd
import numpy as np

In [73]:
#create variables for loading screen
def bouncing_bar():
    width = 20
    pos = 0
    direction = 1

    for _ in range(20):  # number of animation steps
        bar = [" "] * width
        bar[pos] = "#"
        sys.stdout.write("\r[" + "".join(bar) + "]")
        sys.stdout.flush()
        time.sleep(0.001)

        pos += direction
        if pos == 0 or pos == width - 1:
            direction *= -1
    print()
    



In [74]:
print("-----Hello! Welcome to combined_program.ipynb!-----")
print("-----I would like to thank for taking time to use my predition model for Barnegat Bay!-----")
print("-----If you encounter any problems, please let me know!------")
print("-----(This model is only trained for summer (June-August) and will not perform accurately)-----")
print()
print()
print()
print("-----This program will take a csv of your choosing, (make sure it is formatted as specified in the readme)-----")
print("-----return your orginal data (in the same location), a editied version for the model, and a results csv-----")
print("-----Several times you may be asked 'Yes' or 'No' questions and reply with (y/n) in the terminal-----")
print()
print()
print()
while True:
    reply = input("------Are you ready to start? (You have no choice in this one :D)-----").strip().lower()
    if reply in ("y"):
        break
    print("-----Please enter 'y' to continue-----")

bouncing_bar()

-----Hello! Welcome to combined_program.ipynb!-----
-----I would like to thank for taking time to use my predition model for Barnegat Bay!-----
-----If you encounter any problems, please let me know!------
-----(This model is only trained for summer (June-August) and will not perform accurately)-----



-----This program will take a csv of your choosing, (make sure it is formatted as specified in the readme)-----
-----return your orginal data (in the same location), a editied version for the model, and a results csv-----
-----Several times you may be asked 'Yes' or 'No' questions and reply with (y/n) in the terminal-----



[                   #]


In [75]:
#replace the direction column with the corresponding degree values
direction_map = {'N': 0,'NNE': 1,'NE': 2,'ENE': 3,'E': 4,'ESE': 5,'SE': 6,'SSE': 7,
            'S': 8,'SSW': 9,'SW': 10,'WSW': 11,'W': 12,'WNW': 13,'NW': 14,'NNW': 15
        }

In [76]:
#Get all variables
print("-----Please keep water temperature in the same unit (C or F) and air temperature in the same unit (C or F)-----")
print("-----Please enter the following variables from the Stafford Weather Station-----")
data_main_air_temp = float(input("------What is the current air temperature? (C or F)-----"))
bouncing_bar()
data_humidity_per = float(input("------What is the current humidity percentage?-----"))
bouncing_bar()

#replace the direction column with the corresponding degree values
direction_map = {'N': 0,'NNE': 1,'NE': 2,'ENE': 3,'E': 4,'ESE': 5,'SE': 6,'SSE': 7,
            'S': 8,'SSW': 9,'SW': 10,'WSW': 11,'W': 12,'WNW': 13,'NW': 14,'NNW': 15
        }
while True:
    data_wind_direction = (input("------What is the current wind direction in characters (ex. N, SE, NNE)?-----"))
    data_wind_direction = data_wind_direction.strip().upper()
    bouncing_bar()
    if data_wind_direction not in direction_map:
        print("Unknown direction, please enter one of:", ", ".join(direction_map.keys()))
    else:
        data_wind_direction = direction_map[data_wind_direction]
        break
    

print("-----Remapped Directions to Bins...-----")
bouncing_bar()

-----Please keep water temperature in the same unit (C or F) and air temperature in the same unit (C or F)-----
-----Please enter the following variables from the Stafford Weather Station-----
[                   #]
[                   #]
[                   #]
-----Remapped Directions to Bins...-----
[                   #]


In [77]:
#Get variables continued
data_wind_speed = float(input("------What is the current wind speed in mph?-----"))
bouncing_bar()
data_gusting = float(input("------What is the current gusting (max wind speed) in mph?-----"))
bouncing_bar()
data_pressure = float(input("------What is the current atmospheric pressure in IN?-----"))
bouncing_bar()
data_rainfall = float(input("------What is the current rainfall in inches?-----"))
bouncing_bar()


print("-----Please enter the following variables from the SCYC Weather Station-----")
data_lbi_temp = float(input("------What is the current LBI temperature in degrees (C or F)?-----"))
bouncing_bar()

print("-----Please enter the following variables from the nearby water body or the NJDEP MB_01 Buoy-----")
data_bay_temp = float(input("------What is the current bay temperature in degrees (C or F)?-----"))
bouncing_bar()
data_salinity = float(input("------What is the current salinity in ppt?-----"))
bouncing_bar()

print("-----Please enter the following variables from the NDBC Station 44091-----")
data_ocean_temp = float(input("------What is the current ocean temperature in degrees (C or F)?-----"))
bouncing_bar()

[                   #]
[                   #]
[                   #]
[                   #]
-----Please enter the following variables from the SCYC Weather Station-----
[                   #]
-----Please enter the following variables from the nearby water body or the NJDEP MB_01 Buoy-----
[                   #]
[                   #]
-----Please enter the following variables from the NDBC Station 44091-----
[                   #]


In [78]:
#make sure values are in celsius
while True:
    reply = input("-----Are your Air Tempuratures in Celsius (y/n)?").strip().lower()
    if reply in ("y", "n"):
        break
    print("-----Please enter 'y' or 'n'-----")
    
if reply=='n':
    #convert to Celsius
    data_main_air_temp = round((data_main_air_temp-32) * 5.0/9.0, 1)
    data_lbi_temp = round((data_lbi_temp  -32) * 5.0/9.0, 1)
    print("-----Converted air temperatures from Fahrenheit to Celsius...-----")
    bouncing_bar()
    
else:
    print("-----Thanks! Less work for me-----")

reply = input("-----Are your Water Tempuratures in Celsius (y/n)?").strip().lower()
if reply=='n':
    #convert to Celsius
    data_ocean_temp = round((data_ocean_temp-32) * 5.0/9.0, 1)
    data_bay_temp = round((data_bay_temp-32) * 5.0/9.0, 1)
    print("-----Converted water temperatures from Fahrenheit to Celsius...-----")
    bouncing_bar()
else:
    print("-----Thanks! Less work for me-----")


-----Converted air temperatures from Fahrenheit to Celsius...-----
[                   #]
-----Thanks! Less work for me-----


In [79]:
#round all of the columns
data_main_air_temp = round(data_main_air_temp, 1)
data_humidity_per = round(data_humidity_per, 1)
data_wind_speed = round(data_wind_speed, 1)
data_gusting = round(data_gusting, 1)
data_pressure = round(data_pressure, 2)
data_rainfall = round(data_rainfall, 2)
data_bay_temp = round(data_bay_temp, 2)
data_salinity = round(data_salinity, 2)
data_lbi_temp = round(data_lbi_temp, 1)
data_ocean_temp = round(data_ocean_temp, 1)
print("-----Rounded Data...-----")
bouncing_bar()

-----Rounded Data...-----
[                   #]


In [ ]:
#determines if its a onshore breeze and adds a new column for it
onshore_degrees = [8, 6, 7, 4, 3, 2, 1]
if data_wind_direction in onshore_degrees:
    data_onshore_flag = 1
else:
    data_onshore_flag = 0  

print("-----Determined if it's an Onshore Breeze...------")
bouncing_bar()

if data_onshore_flag == 1:
    print("-----It's an Onshore Breeze!-----")
else:    
    print("-----It's not an Onshore Breeze!-----")
bouncing_bar()

-----Determined Onshore Breezes...------
[                   #]
Onshore flag (0 = no, 1 = yes): 1


In [ ]:
#No upwelling can be predicted
print("-----No upwelling can be predicted for only one hour of data-----")
bouncing_bar()